In [ ]:
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage
from typing import TypedDict, Annotated

from IPython.core.display import display_png, Image
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.constants import START, END
from langgraph.graph import StateGraph, add_messages

# =================1. 定义State数据结构，默认Reducer=================
class LLMState(TypedDict):
    # 继承HumanMessage的基类
    # 这个字典有个messages的key,值是一个列表,里面存储消息对象
    messages: Annotated[list[BaseMessage], add_messages] # 追加消息列表的函数

# 创建大模型对象
model = init_chat_model(
    model='deepseek-v4-flash',
    extra_body={
    'thinking': {
        'type':'disabled'
    }
}
)

# 开始节点
def node_1(state: LLMState):
    # 调用大模型,蒋消息列表全部丢给大模型
    response =  model.invoke(state['messages'])
    return {
        # 直接返回response,里面就是消息列表
        "messages":response
    }


# 创建图表构建器
graph_builder = StateGraph(LLMState)

# 注册节点
graph_builder.add_node("node_1", node_1)

# 定义线的连接
graph_builder.add_edge(START, "node_1")
graph_builder.add_edge("node_1", END)

# 创建完整的图表
graph = graph_builder.compile(checkpointer=InMemorySaver())

# 设立config对象
config = RunnableConfig(configurable={'thread_id':'thread_1'})

# 画出工作流
display_png(Image(graph.get_graph().draw_mermaid_png()))

message_list = [
    SystemMessage(content='你是一只猫娘'),
    HumanMessage(content='宝贝')
]

# =================4.调用Graph，传入默认State值=================
result = graph.invoke({"messages": message_list},config=config)
for msg in result['messages']:
    msg.pretty_print()